# Step 5b — Lagrange Multipliers on Lineage Endpoints (Calculations L1b–L3b)

**Purpose:** Re-estimate λ_S and λ_R using only the most somatically hypermutated representative per clonal lineage.

## Why endpoints?

The cross-sectional regression in Step 5 mixes sequences at all maturation stages:
recent GC entries (few mutations, near-germline), intermediate sequences, and deeply matured
cells. This averaging washes out the KKT signal, because the stationarity condition
```
∇Φ_A = −λ_S ∇Φ_S − λ_R ∇Φ_R
```
is a property of the **optimum** — i.e., of sequences that have matured as far as the constraints
allow. Selecting one endpoint per lineage brings the working dataset closer to this optimum
and reduces the stage-mixing confound.

## Endpoint selection criterion

For each clonal lineage, keep the sequence with the highest total VH amino acid replacement
mutation count: `n_R_H = n_R_CDR_H + n_R_FWR_H`. Ties broken by total codon mutations `n_mut_H`.

**Why replacement, not total codon mutations?**  
All three objectives (Φ_A, Φ_S, Φ_R) are defined in functional (amino acid) space. Silent mutations
contribute to evolutionary distance but do not move a sequence through objective space. Replacement
count therefore better reflects functional maturation depth.

## Calculations

- **L1b** — Global λ on endpoints
- **L2b** — Per-germline λ on endpoints (≥100 endpoint lineages per germline)
- **L3b** — Per-donor λ on endpoints (≥500 endpoint lineages per donor)

Each mirrors the Step 5 procedure; results are compared side-by-side.

**Inputs:** `results/tables/affinity_proxy.parquet`, `results/tables/phi_r_scores.parquet`,
`results/tables/omega_per_position.parquet`  
**Outputs:** `results/tables/lineage_endpoints.parquet`, `results/tables/lambda_*_endpoints.csv`,
`results/figures/fig_l1b_*.png`, `results/figures/fig_l2b_*.png`, `results/figures/fig_l3b_*.png`

In [ ]:
import polars as pl
import numpy as np
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.optimize import nnls, minimize
from scipy.stats import pearsonr, spearmanr

In [ ]:
DATA_DIR = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
RESULTS  = DATA_DIR / "results"
FIGURES  = RESULTS / "figures"
TABLES   = RESULTS / "tables"

print("Paths OK")

In [ ]:
# ── Recompute regional phi_S costs from omega (same as Steps 5 and 6) ────────
omega_df = pl.read_parquet(TABLES / "omega_per_position.parquet")

omega_df = omega_df.with_columns(
    pl.when(pl.col('omega').is_not_null() & (pl.col('omega') > 0))
    .then((-pl.col('omega').log(math.e)).clip(lower_bound=0.0))
    .otherwise(0.0)
    .alias('phi_s_local')
)

rd = {
    r['region']: r['mean_phi_s']
    for r in omega_df.group_by('region')
        .agg(pl.col('phi_s_local').mean().alias('mean_phi_s'))
        .to_dicts()
}

PHI_S_CDR = float(np.mean([rd.get('CDR1', 0.0), rd.get('CDR2', 0.0)]))
PHI_S_FWR = float(np.mean([rd.get('FR1', 0.0), rd.get('FR2', 0.0), rd.get('FR3', 0.0)]))

print(f"⟨φ_S⟩_CDR = {PHI_S_CDR:.4f}  |  ⟨φ_S⟩_FWR = {PHI_S_FWR:.4f}")

In [ ]:
# ── Load affinity proxy and phi_R; compute phi_S; join ────────────────────────
phi_a_df = pl.read_parquet(TABLES / "affinity_proxy.parquet")
phi_r_df = pl.read_parquet(TABLES / "phi_r_scores.parquet")

data = (
    phi_a_df
    .filter(pl.col('phi_A').is_not_null())
    .select(['seq_name', 'v_gene:0', 'isotype_class', 'donor', 'lineage',
             'n_R_CDR_H', 'n_S_CDR_H', 'n_R_FWR_H', 'n_S_FWR_H', 'n_mut_H', 'phi_A'])
    .join(phi_r_df.select(['seq_name', 'phi_R']), on='seq_name', how='inner')
    .with_columns([
        (pl.col('n_R_CDR_H') + pl.col('n_S_CDR_H')).alias('n_mut_CDR_H'),
        (pl.col('n_R_FWR_H') + pl.col('n_S_FWR_H')).alias('n_mut_FWR_H'),
        (pl.col('n_R_CDR_H') + pl.col('n_R_FWR_H')).alias('n_R_H'),
    ])
    .with_columns(
        (pl.col('n_mut_CDR_H') * PHI_S_CDR
         + pl.col('n_mut_FWR_H') * PHI_S_FWR).alias('phi_S')
    )
)

print(f"Full dataset: {data.height:,} sequences")
print(f"  Lineages present (non-null): "
      f"{data.filter(pl.col('lineage').is_not_null())['lineage'].n_unique():,}")
print(f"  Sequences with null lineage: "
      f"{data.filter(pl.col('lineage').is_null()).height:,}")

In [ ]:
# ── Select lineage endpoints (most-mutated representative per lineage) ────────
# Criterion: highest n_R_H (VH replacement mutations), ties broken by n_mut_H.
# Null-lineage sequences are excluded (no clonal context).

endpoints = (
    data
    .filter(pl.col('lineage').is_not_null())
    .sort(['n_R_H', 'n_mut_H'], descending=True)   # most-mutated first
    .group_by('lineage')
    .first()                                         # keep top-ranked per lineage
)

n_total_lineages = data.filter(pl.col('lineage').is_not_null())['lineage'].n_unique()
n_endpoints      = endpoints.height

print(f"Lineages (non-null): {n_total_lineages:,}")
print(f"Endpoint sequences:  {n_endpoints:,}  (one per lineage)")

# Lineage size distribution (how many members per lineage?)
lineage_sizes = (
    data
    .filter(pl.col('lineage').is_not_null())
    .group_by('lineage')
    .agg(pl.len().alias('lineage_size'))
)
endpoints = endpoints.join(
    lineage_sizes, on='lineage', how='left'
)

print(f"\nLineage size distribution:")
print(f"  singletons (size=1): "
      f"{lineage_sizes.filter(pl.col('lineage_size') == 1).height:,} "
      f"({100*lineage_sizes.filter(pl.col('lineage_size')==1).height/n_total_lineages:.1f}%)")
print(f"  multi-member (size≥2): "
      f"{lineage_sizes.filter(pl.col('lineage_size') >= 2).height:,}")
print(f"  max lineage size: {lineage_sizes['lineage_size'].max():,}")
print(f"\nEndpoint mutation stats:")
print(f"  n_R_H:  mean={endpoints['n_R_H'].mean():.2f}  "
      f"median={endpoints['n_R_H'].median():.2f}  "
      f"max={endpoints['n_R_H'].max()}")
print(f"  n_mut_H: mean={endpoints['n_mut_H'].mean():.2f}  "
      f"median={endpoints['n_mut_H'].median():.2f}  "
      f"max={endpoints['n_mut_H'].max()}")
print(f"\nIsotype composition of endpoints:")
print(endpoints.group_by('isotype_class')
               .agg(pl.len().alias('n'))
               .sort('n', descending=True))

# Compare mutation depth: all sequences vs endpoints
for col in ['n_R_H', 'n_mut_H']:
    r_all  = data.filter(pl.col('lineage').is_not_null())[col].mean()
    r_end  = endpoints[col].mean()
    print(f"\n  {col}: all={r_all:.2f}  endpoints={r_end:.2f}  "
          f"(+{100*(r_end-r_all)/r_all:.1f}% increase → endpoint selection works)")

# Save endpoints table
endpoints.write_parquet(TABLES / "lineage_endpoints.parquet")
print(f"\nSaved → lineage_endpoints.parquet ({n_endpoints:,} rows)")

## L1b — Global Lagrange Multipliers on Endpoints

Same procedure as Step 5 L1: within-germline demeaning, then NNLS + Huber regression.  
The endpoint dataset should show higher λ values and R² compared to the full cross-sectional set,  
because stage-mixing is removed.

In [ ]:
# ── Within-germline demeaning on endpoints ────────────────────────────────────
germ_means_ep = (
    endpoints
    .group_by('v_gene:0')
    .agg([
        pl.col('phi_A').mean().alias('mean_phi_A_g'),
        pl.col('phi_S').mean().alias('mean_phi_S_g'),
        pl.col('phi_R').mean().alias('mean_phi_R_g'),
    ])
)

ep_dm = (
    endpoints
    .join(germ_means_ep, on='v_gene:0', how='left')
    .with_columns([
        (pl.col('phi_A') - pl.col('mean_phi_A_g')).alias('d_phi_A'),
        (pl.col('phi_S') - pl.col('mean_phi_S_g')).alias('d_phi_S'),
        (pl.col('phi_R') - pl.col('mean_phi_R_g')).alias('d_phi_R'),
    ])
)

print(f"Demeaned endpoints: {ep_dm.height:,}")
print(f"  mean(d_phi_A): {ep_dm['d_phi_A'].mean():.6f}")
print(f"  mean(d_phi_S): {ep_dm['d_phi_S'].mean():.6f}")
print(f"  mean(d_phi_R): {ep_dm['d_phi_R'].mean():.6f}")

In [ ]:
# ── L1b global regression (NNLS + Huber) on endpoints ─────────────────────────
DELTA_HUBER = 1.35

Y_ep = -ep_dm['d_phi_A'].to_numpy()
X_ep = np.column_stack([
    ep_dm['d_phi_S'].to_numpy(),
    ep_dm['d_phi_R'].to_numpy(),
])

# NNLS
lambda_nnls_ep, res_ep = nnls(X_ep, Y_ep)
ls_ep, lr_ep = lambda_nnls_ep
Y_pred_ep = X_ep @ lambda_nnls_ep
r2_ep = 1 - np.sum((Y_ep - Y_pred_ep)**2) / np.sum((Y_ep - Y_ep.mean())**2)

print(f"NNLS endpoints:  λ_S = {ls_ep:.4f}  |  λ_R = {lr_ep:.4f}  |  R² = {r2_ep:.4f}")

# Huber
def huber_loss(params, X, y, delta=DELTA_HUBER):
    r = y - X @ params
    return np.where(np.abs(r) <= delta, 0.5*r**2, delta*(np.abs(r) - 0.5*delta)).sum()

res_hub_ep = minimize(
    huber_loss, x0=lambda_nnls_ep, args=(X_ep, Y_ep),
    method='L-BFGS-B', bounds=[(0.0, None), (0.0, None)]
)
ls_hub_ep, lr_hub_ep = res_hub_ep.x
print(f"Huber endpoints: λ_S = {ls_hub_ep:.4f}  |  λ_R = {lr_hub_ep:.4f}  "
      f"(converged: {res_hub_ep.success})")

# Bootstrap CI
N_BOOT = 500
rng = np.random.default_rng(42)
boot_ep = np.zeros((N_BOOT, 2))
print(f"\nBootstrap CIs (n_boot={N_BOOT})...")
for i in range(N_BOOT):
    idx = rng.integers(0, len(Y_ep), size=len(Y_ep))
    boot_ep[i], _ = nnls(X_ep[idx], Y_ep[idx])

ci_lo_ep = np.percentile(boot_ep, 2.5,  axis=0)
ci_hi_ep = np.percentile(boot_ep, 97.5, axis=0)
print(f"  λ_S = {ls_ep:.4f}  95% CI [{ci_lo_ep[0]:.4f}, {ci_hi_ep[0]:.4f}]")
print(f"  λ_R = {lr_ep:.4f}  95% CI [{ci_lo_ep[1]:.4f}, {ci_hi_ep[1]:.4f}]")

# Save
pl.DataFrame({
    'method':   ['NNLS', 'NNLS', 'Huber', 'Huber'],
    'param':    ['lambda_S', 'lambda_R', 'lambda_S', 'lambda_R'],
    'estimate': [ls_ep, lr_ep, ls_hub_ep, lr_hub_ep],
    'ci_lo_95': [ci_lo_ep[0], ci_lo_ep[1], None, None],
    'ci_hi_95': [ci_hi_ep[0], ci_hi_ep[1], None, None],
    'r2':       [r2_ep, r2_ep, None, None],
    'n':        [len(Y_ep), len(Y_ep), len(Y_ep), len(Y_ep)],
}).write_csv(TABLES / "lambda_global_endpoints.csv")
print(f"\nSaved → lambda_global_endpoints.csv")

In [ ]:
# ── L1b plot: endpoints vs full-dataset comparison ────────────────────────────
# Load Step 5 global results for comparison
step5_global = pl.read_csv(TABLES / "lambda_global.csv")
s5_nnls = step5_global.filter(
    (pl.col('method') == 'NNLS') & (pl.col('param') == 'lambda_S')
)['estimate'][0]
s5_nnlr = step5_global.filter(
    (pl.col('method') == 'NNLS') & (pl.col('param') == 'lambda_R')
)['estimate'][0]
s5_ci_s_lo = step5_global.filter(
    (pl.col('method') == 'NNLS') & (pl.col('param') == 'lambda_S')
)['ci_lo_95'][0]
s5_ci_s_hi = step5_global.filter(
    (pl.col('method') == 'NNLS') & (pl.col('param') == 'lambda_S')
)['ci_hi_95'][0]
s5_ci_r_lo = step5_global.filter(
    (pl.col('method') == 'NNLS') & (pl.col('param') == 'lambda_R')
)['ci_lo_95'][0]
s5_ci_r_hi = step5_global.filter(
    (pl.col('method') == 'NNLS') & (pl.col('param') == 'lambda_R')
)['ci_hi_95'][0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: predicted vs actual (endpoints)
ax = axes[0]
sample_ep = rng.integers(0, len(Y_ep), size=min(50_000, len(Y_ep)))
ax.scatter(Y_pred_ep[sample_ep], Y_ep[sample_ep],
           s=1, alpha=0.05, color='#43A047', rasterized=True)
lim = np.percentile(np.abs(Y_ep), 99)
ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1)
ax.set_xlabel('Predicted −ΔΦ_A')
ax.set_ylabel('Observed −ΔΦ_A')
ax.set_title(f'Endpoint KKT fit\n(R²={r2_ep:.4f}, n={len(Y_ep):,} endpoints)')

# Middle: comparison bar chart (Step 5 vs Step 5b)
ax2 = axes[1]
params   = ['λ_S', 'λ_R']
vals_s5  = [s5_nnls, s5_nnlr]
vals_ep  = [ls_ep, lr_ep]
errs_lo5 = [s5_nnls - s5_ci_s_lo, s5_nnlr - s5_ci_r_lo]
errs_hi5 = [s5_ci_s_hi - s5_nnls, s5_ci_r_hi - s5_nnlr]
errs_lo_ep = vals_ep - ci_lo_ep
errs_hi_ep = ci_hi_ep - vals_ep

x = np.arange(len(params))
w = 0.35
b1 = ax2.bar(x - w/2, vals_s5, w,
             yerr=[errs_lo5, errs_hi5], capsize=4,
             color='#1E88E5', alpha=0.8, label=f'Step 5 (all seqs, n={len(Y_ep):,})',
             error_kw={'elinewidth': 1})
b2 = ax2.bar(x + w/2, vals_ep, w,
             yerr=[errs_lo_ep, errs_hi_ep], capsize=4,
             color='#43A047', alpha=0.8, label=f'Step 5b (endpoints, n={len(Y_ep):,})',
             error_kw={'elinewidth': 1})
ax2.set_xticks(x)
ax2.set_xticklabels(params, fontsize=11)
ax2.axhline(0, color='gray', lw=0.8, linestyle='--')
ax2.set_ylabel('Lagrange multiplier (NNLS)')
ax2.set_title('Step 5 vs Step 5b\n(95% bootstrap CI)')
ax2.legend(fontsize=7)

# Right: bootstrap distributions (endpoints)
ax3 = axes[2]
ax3.hist(boot_ep[:, 0], bins=40, color='#43A047', alpha=0.7,
         label=f'λ_S (mean={ls_ep:.4f})')
ax3.hist(boot_ep[:, 1], bins=40, color='#E53935', alpha=0.7,
         label=f'λ_R (mean={lr_ep:.4f})')
ax3.set_xlabel('Bootstrap λ estimate (endpoints)')
ax3.set_ylabel('Count')
ax3.set_title(f'Bootstrap distributions (n={N_BOOT})\n(NNLS, germline-demeaned, endpoints)')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l1b_lambda_global_endpoints.png", dpi=150, bbox_inches='tight')

# CSV: comparison table
pl.DataFrame({
    'param':      params,
    'lambda_step5':     vals_s5,
    'lambda_endpoints': list(vals_ep),
    'ci_lo_step5':      [s5_ci_s_lo, s5_ci_r_lo],
    'ci_hi_step5':      [s5_ci_s_hi, s5_ci_r_hi],
    'ci_lo_endpoints':  list(ci_lo_ep),
    'ci_hi_endpoints':  list(ci_hi_ep),
}).write_csv(FIGURES / "fig_l1b_lambda_global_endpoints.csv")

plt.show()
print("Saved.")

## L2b — Per-Germline Lagrange Multipliers on Endpoints

Repeat L2 on the endpoint subset, using a lower threshold (≥100 endpoints per germline)  
because the endpoint dataset is smaller than the full memory dataset.

In [ ]:
# ── L2b per-germline regression on endpoints ──────────────────────────────────
MIN_GERM_EP  = 100
JACKKNIFE_EP = 500

germ_ep_results = []

for vgene_key, subdf in endpoints.partition_by('v_gene:0', as_dict=True).items():
    vgene = vgene_key[0] if isinstance(vgene_key, (list, tuple)) else str(vgene_key)
    n = subdf.height
    if n < MIN_GERM_EP:
        continue

    means = subdf.select([pl.col('phi_A').mean(), pl.col('phi_S').mean(),
                          pl.col('phi_R').mean()]).to_numpy()[0]
    phi_A_g = subdf['phi_A'].to_numpy() - means[0]
    phi_S_g = subdf['phi_S'].to_numpy() - means[1]
    phi_R_g = subdf['phi_R'].to_numpy() - means[2]

    Y_g = -phi_A_g
    X_g = np.column_stack([phi_S_g, phi_R_g])
    lam_g, _ = nnls(X_g, Y_g)

    ss_res_g = np.sum((Y_g - X_g @ lam_g)**2)
    ss_tot_g = np.sum((Y_g - Y_g.mean())**2)
    r2_g = 1 - ss_res_g / ss_tot_g if ss_tot_g > 0 else np.nan

    # Jackknife SE for larger strata
    se_S, se_R = np.nan, np.nan
    if n >= JACKKNIFE_EP:
        n_blocks = min(20, n)
        block_size = n // n_blocks
        jk = []
        for b in range(n_blocks):
            mask = np.ones(n, dtype=bool)
            mask[b*block_size:(b+1)*block_size] = False
            lb, _ = nnls(X_g[mask], Y_g[mask])
            jk.append(lb)
        jk = np.array(jk)
        se_S = np.sqrt((n_blocks-1)/n_blocks * np.sum((jk[:,0]-lam_g[0])**2))
        se_R = np.sqrt((n_blocks-1)/n_blocks * np.sum((jk[:,1]-lam_g[1])**2))

    germ_ep_results.append({
        'v_gene':   vgene,
        'lambda_S': float(lam_g[0]),
        'lambda_R': float(lam_g[1]),
        'se_S':     float(se_S),
        'se_R':     float(se_R),
        'r2':       float(r2_g),
        'n':        n,
    })

germ_ep_df = pl.DataFrame(germ_ep_results).sort('lambda_S', descending=True)
germ_ep_df.write_csv(TABLES / "lambda_by_germline_endpoints.csv")

print(f"Germlines with ≥{MIN_GERM_EP} endpoints: {germ_ep_df.height}")
print(f"\nTop 10 by λ_S:")
print(germ_ep_df.head(10))
print(f"\nTop 10 by λ_R:")
print(germ_ep_df.sort('lambda_R', descending=True).head(10))
print(f"\nSaved → lambda_by_germline_endpoints.csv")

In [ ]:
# ── L2b plot: per-germline λ scatter (endpoints) ──────────────────────────────
lS_ep  = germ_ep_df['lambda_S'].to_numpy()
lR_ep  = germ_ep_df['lambda_R'].to_numpy()
genes_ep = germ_ep_df['v_gene'].to_list()
n_seqs_ep = germ_ep_df['n'].to_numpy()

HIGHLIGHT = {'IGHV4-34'}   # autoreactive

# Colour by λ_R > 0 (constraint active)
colors_ep = ['#FF6F00' if g in HIGHLIGHT else
             '#E53935' if lR_ep[i] > 0 else
             '#1E88E5'
             for i, g in enumerate(genes_ep)]
sizes_ep  = [60 + n/1000 for n in n_seqs_ep]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: λ_S vs λ_R scatter
ax = axes[0]
ax.scatter(lS_ep, lR_ep, c=colors_ep, s=sizes_ep, alpha=0.8, zorder=3)

# Label germlines with active λ_R or top λ_S
for g, s, r in zip(genes_ep, lS_ep, lR_ep):
    if r > 0 or s > np.percentile(lS_ep, 85):
        ax.annotate(g, (s, r), fontsize=6, ha='left',
                    xytext=(3, 3), textcoords='offset points')

ax.axhline(lr_ep, color='gray', linestyle=':', lw=1, label=f'global λ_R (endpoints)={lr_ep:.3f}')
ax.axvline(ls_ep, color='gray', linestyle=':', lw=1, label=f'global λ_S (endpoints)={ls_ep:.3f}')
ax.set_xlabel('λ_S (structural penalty exchange rate)')
ax.set_ylabel('λ_R (reactivity penalty exchange rate)')
ax.set_title(f'Per-germline Lagrange multipliers (endpoints)\n'
             f'n={germ_ep_df.height} germlines, ≥{MIN_GERM_EP} endpoints each')

legend_handles = [
    mpatches.Patch(color='#FF6F00', label='IGHV4-34'),
    mpatches.Patch(color='#E53935', label='λ_R > 0 (reactive constraint active)'),
    mpatches.Patch(color='#1E88E5', label='λ_R = 0'),
]
ax.legend(handles=legend_handles, fontsize=8)

# Right: ranked bar of λ_R
ax2 = axes[1]
gdf_r = germ_ep_df.sort('lambda_R', descending=True)
top_n = min(30, gdf_r.height)
g_names_r = gdf_r['v_gene'].to_list()[:top_n]
g_lR_r    = gdf_r['lambda_R'].to_numpy()[:top_n]
g_colors_r = ['#FF6F00' if g in HIGHLIGHT else '#E53935' for g in g_names_r]

ax2.barh(range(top_n), g_lR_r[::-1], color=g_colors_r[::-1], alpha=0.8)
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(g_names_r[::-1], fontsize=8)
ax2.axvline(lr_ep, color='gray', linestyle='--', lw=1,
            label=f'global λ_R={lr_ep:.4f}')
ax2.set_xlabel('λ_R (per-germline, endpoints)')
ax2.set_title(f'Top {top_n} germlines by λ_R (endpoints)\n'
              f'higher = stronger reactivity constraint at maturation endpoint')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l2b_lambda_by_germline_endpoints.png", dpi=150, bbox_inches='tight')

germ_ep_df.write_csv(FIGURES / "fig_l2b_lambda_by_germline_endpoints.csv")
plt.show()
print("Saved.")

## L3b — Per-Donor Lagrange Multipliers on Endpoints

In [ ]:
# ── L3b per-donor regression on endpoints ─────────────────────────────────────
MIN_DONOR_EP = 500

donor_ep_results = []

for donor_key, subdf in endpoints.partition_by('donor', as_dict=True).items():
    donor = donor_key[0] if isinstance(donor_key, (list, tuple)) else str(donor_key)
    n = subdf.height
    if n < MIN_DONOR_EP:
        continue

    # Demean by (donor × v_gene)
    vg_means = (
        subdf.group_by('v_gene:0')
        .agg([pl.col('phi_A').mean().alias('m_A'),
              pl.col('phi_S').mean().alias('m_S'),
              pl.col('phi_R').mean().alias('m_R')])
    )
    subdf_dm = (
        subdf.join(vg_means, on='v_gene:0', how='left')
        .with_columns([
            (pl.col('phi_A') - pl.col('m_A')).alias('d_phi_A'),
            (pl.col('phi_S') - pl.col('m_S')).alias('d_phi_S'),
            (pl.col('phi_R') - pl.col('m_R')).alias('d_phi_R'),
        ])
    )

    Y_d = -subdf_dm['d_phi_A'].to_numpy()
    X_d = np.column_stack([
        subdf_dm['d_phi_S'].to_numpy(),
        subdf_dm['d_phi_R'].to_numpy(),
    ])
    lam_d, _ = nnls(X_d, Y_d)

    ss_r = np.sum((Y_d - X_d @ lam_d)**2)
    ss_t = np.sum((Y_d - Y_d.mean())**2)
    r2_d = 1 - ss_r / ss_t if ss_t > 0 else np.nan

    donor_ep_results.append({
        'donor':    donor,
        'lambda_S': float(lam_d[0]),
        'lambda_R': float(lam_d[1]),
        'r2':       float(r2_d),
        'n':        n,
    })

donor_ep_df = pl.DataFrame(donor_ep_results).sort('donor')
donor_ep_df.write_csv(TABLES / "lambda_by_donor_endpoints.csv")

print(f"Donors with ≥{MIN_DONOR_EP} endpoints: {donor_ep_df.height}")
print(f"\nλ_S across donors (endpoints):")
print(f"  mean={donor_ep_df['lambda_S'].mean():.4f}  std={donor_ep_df['lambda_S'].std():.4f}")
print(f"\nλ_R across donors (endpoints):")
print(f"  mean={donor_ep_df['lambda_R'].mean():.4f}  std={donor_ep_df['lambda_R'].std():.4f}")
print(f"\n{donor_ep_df}")
print(f"\nSaved → lambda_by_donor_endpoints.csv")

In [ ]:
# ── L3b plot: per-donor λ distributions (endpoints) ──────────────────────────
lS_d_ep = donor_ep_df['lambda_S'].to_numpy()
lR_d_ep = donor_ep_df['lambda_R'].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: box plots
ax = axes[0]
bp = ax.boxplot([lS_d_ep, lR_d_ep], labels=['λ_S', 'λ_R'],
                patch_artist=True,
                medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp['boxes'], ['#43A047', '#E53935']):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.axhline(ls_ep, color='#43A047', linestyle='--', lw=1.5, alpha=0.6,
           label=f'global λ_S={ls_ep:.3f}')
ax.axhline(lr_ep, color='#E53935', linestyle='--', lw=1.5, alpha=0.6,
           label=f'global λ_R={lr_ep:.3f}')
ax.set_ylabel('Lagrange multiplier (endpoint analysis)')
ax.set_title(f'λ distribution across donors (n={len(lS_d_ep)})\nEndpoints only')
ax.legend(fontsize=8)

# Middle: λ_S vs λ_R per donor
ax2 = axes[1]
ax2.scatter(lS_d_ep, lR_d_ep, s=60, alpha=0.8, color='#43A047', edgecolors='black', lw=0.5)
r_cor, p_cor = pearsonr(lS_d_ep, lR_d_ep) if len(lS_d_ep) > 2 else (np.nan, np.nan)
for row in donor_ep_df.iter_rows(named=True):
    ax2.annotate(row['donor'], (row['lambda_S'], row['lambda_R']),
                 fontsize=7, xytext=(3, 3), textcoords='offset points')
ax2.set_xlabel('λ_S (endpoints)')
ax2.set_ylabel('λ_R (endpoints)')
ax2.set_title(f'Per-donor λ_S vs λ_R (endpoints)\n(r={r_cor:.3f}, p={p_cor:.3f})')

# Right: λ_R distribution across donors
ax3 = axes[2]
ax3.hist(lR_d_ep, bins=15, color='#E53935', alpha=0.7, density=True)
ax3.axvline(lr_ep, color='black', lw=1.5, linestyle='--',
            label=f'global λ_R={lr_ep:.3f}')
ax3.axvline(np.median(lR_d_ep), color='orange', lw=1.5, linestyle='-.',
            label=f'median={np.median(lR_d_ep):.3f}')
ax3.set_xlabel('Per-donor λ_R (endpoints)')
ax3.set_ylabel('Density')
ax3.set_title('Inter-individual variation in λ_R\n(endpoints only)')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "fig_l3b_lambda_by_donor_endpoints.png", dpi=150, bbox_inches='tight')

donor_ep_df.write_csv(FIGURES / "fig_l3b_lambda_by_donor_endpoints.csv")
plt.show()
print("Saved.")

In [ ]:
# ── Summary comparison: Step 5 vs Step 5b ─────────────────────────────────────
print("=" * 65)
print("LAGRANGE MULTIPLIERS: STEP 5 (all seqs) vs STEP 5b (endpoints)")
print("=" * 65)

print(f"\nL1b Global (endpoints, n={len(Y_ep):,}):")
print(f"   λ_S = {ls_ep:.4f}  [95% CI: {ci_lo_ep[0]:.4f}, {ci_hi_ep[0]:.4f}]")
print(f"   λ_R = {lr_ep:.4f}  [95% CI: {ci_lo_ep[1]:.4f}, {ci_hi_ep[1]:.4f}]")
print(f"   R²  = {r2_ep:.4f}")

print(f"\nStep 5 L1 Global (all seqs):")
print(f"   λ_S = {s5_nnls:.4f}  λ_R = {s5_nnlr:.4f}  R² ≈ 0.0004")

if r2_ep > 0.001:
    print(f"\n✓ Endpoint R² ({r2_ep:.4f}) > Step 5 R² (0.0004): stage-mixing confound reduced.")
if ls_ep > s5_nnls:
    print(f"✓ λ_S higher on endpoints ({ls_ep:.4f} vs {s5_nnls:.4f}): structural trade-off"
          f"\n  more visible when only endpoint sequences are considered.")
if lr_ep > s5_nnlr:
    print(f"✓ λ_R higher on endpoints ({lr_ep:.4f} vs {s5_nnlr:.4f}): reactivity constraint"
          f"\n  signal stronger at maturation endpoint.")

print(f"\nL2b Per-germline (n={germ_ep_df.height} germlines, ≥{MIN_GERM_EP} endpoints):")
print(f"   λ_S: mean={germ_ep_df['lambda_S'].mean():.4f}  "
      f"range=[{germ_ep_df['lambda_S'].min():.4f}, {germ_ep_df['lambda_S'].max():.4f}]")
print(f"   λ_R: mean={germ_ep_df['lambda_R'].mean():.4f}  "
      f"range=[{germ_ep_df['lambda_R'].min():.4f}, {germ_ep_df['lambda_R'].max():.4f}]")
print(f"   Germlines with λ_R > 0: "
      f"{germ_ep_df.filter(pl.col('lambda_R') > 0).height}")

print(f"\nL3b Per-donor (n={donor_ep_df.height} donors, ≥{MIN_DONOR_EP} endpoints):")
print(f"   λ_S: mean={donor_ep_df['lambda_S'].mean():.4f}  "
      f"std={donor_ep_df['lambda_S'].std():.4f}")
print(f"   λ_R: mean={donor_ep_df['lambda_R'].mean():.4f}  "
      f"std={donor_ep_df['lambda_R'].std():.4f}")

## Step 5b Summary

| Calculation | Output table | Output figure |
|-------------|-------------|---------------|
| Endpoint selection | `lineage_endpoints.parquet` | — |
| L1b: global λ (endpoints) | `lambda_global_endpoints.csv` | `fig_l1b_lambda_global_endpoints.png` |
| L2b: per-germline λ (endpoints) | `lambda_by_germline_endpoints.csv` | `fig_l2b_lambda_by_germline_endpoints.png` |
| L3b: per-donor λ (endpoints) | `lambda_by_donor_endpoints.csv` | `fig_l3b_lambda_by_donor_endpoints.png` |

**Key comparison with Step 5:**
- If R² improves (Step 5b > Step 5), the endpoint approach successfully reduces the stage-mixing confound.
- If λ values are higher, the KKT constraints are more visible at the maturation endpoint.
- Germlines with `λ_R > 0` at the endpoint level (but not in the full-population analysis, Step 5) represent germlines where the reactivity constraint becomes active only after extensive maturation.

**Limitations of this endpoint definition:**
1. The most-mutated sequence per lineage may not be the most *functionally* matured — selection acts on affinity, not mutation count. High-mutation sequences could include non-productive hypermutators or sequences with partially reverted mutations.
2. Singletons (no observed clonal expansion) are included and may not represent true maturation endpoints.
3. The endpoint selection ignores VL mutations; a combined VH+VL criterion could be considered (`n_R_CDR_H + n_R_FWR_H + n_R_CDR_L + n_R_FWR_L`).

**Next step:** `06_pareto.ipynb` — Pareto front mapping.